# Explicabilité — Niveau 1 (par foyer) et Niveau 2 (typologie de ménages)

## Objectif

Reproduire l'approche surrogate du TFC (XGBoost + SHAP + arbre de décision,
`04_UCI_clustering.pdf` section 1.10-1.12) :
- **Niveau 1** : pour chacun des 20 foyers, quelle variable domine la distinction entre
  régimes journaliers ? (attendu : `is_weekend`, comme sur le foyer UCI)
- **Niveau 2** : pour la typologie de ménages (n=20), quelle variable domine la distinction
  entre types de foyers ? (attendu : probablement `mean_load` ou `load_factor`, puisque
  `is_weekend` s'annule en moyenne sur une période longue — à vérifier empiriquement ici)

La comparaison Niveau 1 vs Niveau 2 est un résultat à part entière de l'article.

**Pré-requis :** `pip install xgboost shap` si ces bibliothèques ne sont pas déjà installées
(elles l'étaient dans l'environnement utilisé pour le TFC original).

AUTEUR : ENANGA DANIEL CHRISTIAN


## Étape 0 : Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
matplotlib.rcParams['figure.dpi'] = 120

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(BASE_DIR, "src"))
from features import FEATURES_FINAL

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "refit")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
IMAGES_DIR = os.path.join(BASE_DIR, "images", "explainability")
os.makedirs(IMAGES_DIR, exist_ok=True)

RANDOM_STATE = 42
print("Bibliothèques chargées (XGBoost, SHAP).")


## Étape 1 : Explicabilité Niveau 1 — boucle sur les 20 foyers

Pour chaque foyer, un XGBoost est entraîné à reproduire les labels de cluster du Niveau 1
(Notebook 03), puis le gain (%) attribué à `is_weekend` est extrait.

In [ ]:
def scaled_tree_params(n_samples: int) -> dict:
    """
    Calcule des hyperparamètres d'arbre de décision (min_samples_leaf,
    min_samples_split) PROPORTIONNELS à la taille de l'échantillon, à
    partir des ratios utilisés dans le TFC original sur le foyer UCI
    (min_samples_leaf=50, min_samples_split=100, pour n=1431 jours, soit
    ~3.5% et ~7% de n). Ceci évite le bug consistant à réutiliser des
    valeurs ABSOLUES calibrées pour n=1431 sur des foyers REFIT (~600
    jours) ou pire, sur la typologie de ménages (n=20 seulement) -- où
    min_samples_split=100 rendrait tout arbre exclusivement racine.
    """
    leaf_ratio, split_ratio = 50 / 1431, 100 / 1431
    return {
        "min_samples_leaf": max(2, int(round(leaf_ratio * n_samples))),
        "min_samples_split": max(4, int(round(split_ratio * n_samples))),
    }


def explain_house(X_scaled: np.ndarray, labels: np.ndarray, feature_names, random_state=42) -> dict:
    """Entraîne un XGBoost surrogate (mêmes hyperparamètres que le TFC
    original, 04_UCI_clustering.pdf) et retourne l'accuracy + l'importance
    (gain) de chaque feature, normalisée en pourcentage du gain total."""
    if len(np.unique(labels)) < 2:
        return None

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, labels, test_size=0.3, random_state=random_state, stratify=labels
    )
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=random_state, eval_metric="mlogloss", use_label_encoder=False,
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    booster = model.get_booster()
    gain_dict = booster.get_score(importance_type="gain")
    # Les clés XGBoost sont 'f0', 'f1', ... -> remapper vers les noms de features
    gains = {feature_names[int(k[1:])]: v for k, v in gain_dict.items()}
    total_gain = sum(gains.values()) if gains else 1.0
    gains_pct = {f: 100 * gains.get(f, 0.0) / total_gain for f in feature_names}

    return {"accuracy": acc, "gains_pct": gains_pct, "model": model}


level1_labels_path = os.path.join(RESULTS_DIR, "level1_cluster_labels.csv")
level1_features_path = os.path.join(PROCESSED_DIR, "level1_features_raw.csv")

explain_l1_rows = []
if os.path.exists(level1_labels_path) and os.path.exists(level1_features_path):
    df_labels = pd.read_csv(level1_labels_path, parse_dates=["date"])
    df_features_raw = pd.read_csv(level1_features_path, parse_dates=["date"])
    df_merged = df_features_raw.merge(df_labels, on=["house_id", "date"])

    for house_id in sorted(df_merged["house_id"].unique()):
        df_h = df_merged[df_merged["house_id"] == house_id]
        X_scaled = StandardScaler().fit_transform(df_h[FEATURES_FINAL].values)
        result = explain_house(X_scaled, df_h["cluster"].values, FEATURES_FINAL, random_state=RANDOM_STATE)
        if result is None:
            continue
        row = {"house_id": house_id, "accuracy": result["accuracy"]}
        row.update({f"gain_pct_{k}": v for k, v in result["gains_pct"].items()})
        explain_l1_rows.append(row)

    df_explain_l1 = pd.DataFrame(explain_l1_rows)
    print(df_explain_l1.round(2).to_string(index=False))
else:
    print("[ATTENTION] Fichiers Niveau 1 introuvables -- exécutez d'abord les Notebooks 02 et 03.")
    df_explain_l1 = pd.DataFrame()


## Étape 2 : `is_weekend` domine-t-il partout ? (comparaison inter-foyers)

Sur le foyer UCI (TFC original), `is_weekend` représentait **87% du gain XGBoost total**.
La figure ci-dessous vérifie si ce résultat se généralise aux 20 foyers REFIT.

In [ ]:
UCI_IS_WEEKEND_GAIN_PCT = 87.0  # TFC original, Tab. 24

if not df_explain_l1.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(df_explain_l1["house_id"].astype(str), df_explain_l1["gain_pct_is_weekend"], color="#2E86AB")
    ax.axhline(UCI_IS_WEEKEND_GAIN_PCT, color="#C73E1D", linestyle="--",
               label=f"Référence UCI ({UCI_IS_WEEKEND_GAIN_PCT:.0f}%)")
    ax.set_title("Part du gain XGBoost attribuée à is_weekend — Niveau 1, par foyer", fontweight="bold")
    ax.set_ylabel("Gain is_weekend (%)")
    ax.set_xlabel("Foyer REFIT")
    plt.xticks(rotation=60, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "01_is_weekend_gain_par_foyer.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nGain moyen is_weekend (REFIT) : {df_explain_l1['gain_pct_is_weekend'].mean():.1f}%")
    print(f"Gain moyen is_weekend (UCI, référence) : {UCI_IS_WEEKEND_GAIN_PCT:.1f}%")


## Étape 2bis : Importance de TOUTES les features (pas seulement is_weekend), Niveau 1

Vue complète (pas seulement is_weekend) : gain XGBoost moyen de chaque feature, sur les 20 foyers,
avec sa dispersion inter-foyers (barre d'erreur = écart-type).

In [ ]:
if not df_explain_l1.empty:
    gain_cols_l1 = [c for c in df_explain_l1.columns if c.startswith("gain_pct_")]
    means_l1 = df_explain_l1[gain_cols_l1].mean().rename(lambda c: c.replace("gain_pct_", ""))
    stds_l1 = df_explain_l1[gain_cols_l1].std().rename(lambda c: c.replace("gain_pct_", ""))
    order = means_l1.sort_values(ascending=True).index

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(order, means_l1[order], xerr=stds_l1[order], color="#2E86AB", capsize=3)
    ax.set_title("Importance moyenne des features (gain XGBoost) — Niveau 1, moyenne sur 20 foyers", fontweight="bold")
    ax.set_xlabel("Gain (%)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "01bis_importance_toutes_features_niveau1.png"), dpi=300, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : 01bis_importance_toutes_features_niveau1.png")


## Étape 3 : Explicabilité Niveau 2 — typologie de ménages (n=20)

Même approche, appliquée cette fois à la classification des **ménages** (pas des journées).

In [ ]:
level2_path = os.path.join(RESULTS_DIR, "level2_household_signatures.csv")

if os.path.exists(level2_path):
    df_household = pd.read_csv(level2_path)
    X_l2_scaled = StandardScaler().fit_transform(df_household[FEATURES_FINAL].values)
    labels_l2 = df_household["cluster"].values

    result_l2 = explain_house(X_l2_scaled, labels_l2, FEATURES_FINAL, random_state=RANDOM_STATE)
    if result_l2 is not None:
        print(f"Accuracy du surrogate XGBoost (Niveau 2) : {result_l2['accuracy']:.4f}")
        print("\nImportance des features (% du gain total) :")
        for feat, pct in sorted(result_l2["gains_pct"].items(), key=lambda x: -x[1]):
            print(f"  {feat:20s} : {pct:5.1f}%")

        # --- Graphique d'importance (Niveau 2) ---
        importance_l2 = pd.Series(result_l2["gains_pct"]).sort_values(ascending=True)
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(importance_l2.index, importance_l2.values, color="#C73E1D")
        ax.set_title("Importance des features (gain XGBoost) — Niveau 2 (typologie de ménages)", fontweight="bold")
        ax.set_xlabel("Gain (%)")
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.savefig(os.path.join(IMAGES_DIR, "02a_importance_niveau2.png"), dpi=300, bbox_inches="tight")
        plt.show()

        # --- SHAP (corrigé pour le cas multi-classes : k=5 clusters au Niveau 2) ---
        # NOTE DE RIGUEUR : shap.summary_plot() attend normalement UNE matrice
        # (n_échantillons x n_features) pour un problème binaire. Pour un problème
        # multi-classes (ici 5 clusters), TreeExplainer retourne soit une LISTE
        # d'une matrice par classe, soit un tableau 3D selon la version de SHAP.
        # Passer cet objet tel quel à summary_plot() produit un graphique confus
        # (plusieurs panneaux mal alignés). Le type "bar" gère nativement le
        # multi-classes et affiche une barre empilée par classe -- lisible et correct.
        explainer = shap.TreeExplainer(result_l2["model"])
        shap_values = explainer.shap_values(X_l2_scaled)

        shap.summary_plot(shap_values, X_l2_scaled, feature_names=FEATURES_FINAL,
                           plot_type="bar", show=False)
        plt.title("SHAP Summary (multi-classes) — Niveau 2 (typologie de ménages)", fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(IMAGES_DIR, "02b_shap_summary_niveau2.png"), dpi=600, bbox_inches="tight")
        plt.show()
        print("Figure sauvegardée : 02b_shap_summary_niveau2.png (barres empilées par cluster)")
    else:
        print("Pas assez de clusters distincts pour entraîner un surrogate au Niveau 2.")
else:
    print("[ATTENTION] level2_household_signatures.csv introuvable -- exécutez d'abord le Notebook 04.")


## Étape 4 : Arbre de décision — règle interprétable pour le Niveau 2

In [ ]:
if os.path.exists(level2_path) and 'result_l2' in dir() and result_l2 is not None:
    tree_params = scaled_tree_params(len(labels_l2))
    tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE, criterion="gini", **tree_params)
    tree.fit(X_l2_scaled, labels_l2)
    tree_acc = accuracy_score(labels_l2, tree.predict(X_l2_scaled))

    print(f"Hyperparamètres adaptés à n={len(labels_l2)} (voir scaled_tree_params) : {tree_params}")
    print(f"Accuracy de l'arbre de décision (Niveau 2, max_depth=3) : {tree_acc:.4f}\n")
    print(export_text(tree, feature_names=FEATURES_FINAL))

    importances = pd.Series(tree.feature_importances_, index=FEATURES_FINAL).sort_values(ascending=False)
    print("\nImportance des features (arbre de décision) :")
    print(importances.round(4))


## Étape 4bis : Visualisation de l'arbre (ce qui manquait pour vraiment "le voir")

`export_text` donne la règle en texte ; voici la même règle sous forme de schéma, directement
comparable à la Figure 25 du TFC original (foyer UCI).

In [ ]:
if os.path.exists(level2_path) and 'tree' in dir():
    class_names = [f"Cluster {c}" for c in sorted(np.unique(labels_l2))]

    fig, ax = plt.subplots(figsize=(18, 9))
    plot_tree(
        tree, feature_names=FEATURES_FINAL, class_names=class_names,
        filled=True, rounded=True, fontsize=10, ax=ax, proportion=False,
    )
    ax.set_title("Arbre de décision — Règles de classification des clusters de ménages (Niveau 2)",
                 fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(IMAGES_DIR, "03_arbre_decision_niveau2.png"), dpi=300, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : 03_arbre_decision_niveau2.png")


## Étape 5 : Comparaison Niveau 1 vs Niveau 2

In [ ]:
if not df_explain_l1.empty and os.path.exists(level2_path) and result_l2 is not None:
    gain_cols = [c for c in df_explain_l1.columns if c.startswith("gain_pct_")]
    l1_mean_importance = df_explain_l1[gain_cols].mean().rename(
        lambda c: c.replace("gain_pct_", "")
    )
    l2_importance = pd.Series(result_l2["gains_pct"])

    comparison = pd.DataFrame({
        "Niveau 1 (moyenne sur 20 foyers, %)": l1_mean_importance,
        "Niveau 2 (typologie ménages, %)": l2_importance,
    }).round(1)
    comparison = comparison.sort_values("Niveau 1 (moyenne sur 20 foyers, %)", ascending=False)
    print(comparison.to_string())

    comparison.to_csv(os.path.join(RESULTS_DIR, "explainability_comparison_l1_l2.csv"))
    print(f"\nTableau comparatif sauvegardé : results/explainability_comparison_l1_l2.csv")


## Étape 6 : Sauvegarde

In [ ]:
if not df_explain_l1.empty:
    df_explain_l1.to_csv(os.path.join(RESULTS_DIR, "level1_explainability.csv"), index=False)
    print("Résultats d'explicabilité Niveau 1 sauvegardés.")


## Résumé final

**Fichiers produits :**
- `results/level1_explainability.csv` — gain is_weekend par foyer (Niveau 1)
- `results/explainability_comparison_l1_l2.csv` — comparaison des deux niveaux
- `images/explainability/01_is_weekend_gain_par_foyer.png`, `02_shap_summary_niveau2.png`

**Étape suivante :** Notebook `06_Robustness_Tests.ipynb` — ablation, bruit, agrégation
temporelle et robustesse du Niveau 2 à la durée d'historique disponible.
